---

Image datasets and measurement

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImageDatabaseWithManifest, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter, pglParameterBatch
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()

---

Load the image database

---

In [ ]:
# load the database of images. Will check in directory for image formats that PIL
# knows about and make a list. This does not load the images, or check to see if they are valid
#imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000")
imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",filenameColumn="image_filename",indexColumn="test_image_nr",captionColumn="concept")

---

Print and display images

---

In [ ]:
# display a single image
#imdb.images[111].display()

# print image metadata one-by-one, this may take some time because it 
# has to open each file 
#imdb.print() 
imdb.print()

---

Display image dataset in a dialog

---

In [ ]:
pgl.traitsDialog(imdb)

In [ ]:
img=imdb.getImage(0)

---

Make a task to display images

---

In [ ]:
class pglImageTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Image Task"
        self.settings.nTrials = 2
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            #'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000",
            #'manifestColumnNames': {'filenameColumn':"filename",'indexColumn':"index",'captionColumn':"caption_1"},
            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",
            'manifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"test_image_nr",'captionColumn':"concept"},
            'imdb': None,
            'nImages': 10,
            'imdbCatch': None,
            'catchImagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_catch_img_12reps",
            'catchManifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"catch_nr",'captionColumn':"original_filename"},
            'nCatchImages': 3,
            'imageSize': 18,
            'nImagesPerTrial': 10,
            'catchTrialEvery': None,
        }        
        p = self.settings.fixedParameters
        
        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5] * p['nImagesPerTrial']

        # initialize image database using parameters set from fixedParameters
        imdb = pglImageDatabaseWithManifest(
            p['imagesDirectory'],
            filenameColumn=p['manifestColumnNames']['filenameColumn'],
            indexColumn=p['manifestColumnNames']['indexColumn'],
            captionColumn=p['manifestColumnNames']['captionColumn'],
        )
        if imdb.nImages==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdb'] = imdb
        
        # initialize catch image database using parameters set from fixedParameters
        imdbCatch = pglImageDatabaseWithManifest(
            p['catchImagesDirectory'],
            filenameColumn=p['catchManifestColumnNames']['filenameColumn'],
            indexColumn=p['catchManifestColumnNames']['indexColumn'],
            captionColumn=p['catchManifestColumnNames']['captionColumn'],
        )
        if imdbCatch.images==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdbCatch'] = imdbCatch
        
        # preload images
        for iImage in range(p['nImages']):
            imdb.preloadImage(iImage)
        for iImage in range(p['nCatchImages']):
            imdbCatch.preloadImage(iImage)
            
        # add parameter for image number
        imageNum = pglParameterBatch('imageNum',np.arange(p['nImages']),batchSize=p['nImagesPerTrial'], catchTrialEvery=p['catchTrialEvery'])
        self.addParameter(imageNum)
        
        # set current image
        self.state.currentImage = None

    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment % 2 == 0: 
            # get image database
            imdb = self.settings.fixedParameters['imdb']
            # get the current image number
            imageNums = self.currentParams['imageNum']
            if imageNums:
                # load an image
                imageNum = imageNums[int(self.state.currentSegment/2)]
                # get the image data
                img = imdb.getImage(imageNum)
                img.convert("RGB")
                print(f"img: {img}")
                # turn into a pglImage
                self.state.currentImage = self.pgl.imageCreate(np.array(img))
            else:
                self.state.currentImage = None
    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment % 2 == 0: 
            if self.state.currentImage:
                self.state.currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

        


---

Setup experiment

---

In [ ]:
pgl.cleanUp()
#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='imageTask')

imageTask = pglImageTask(pgl)
e.addTask(imageTask)

---

run experiment

---

In [ ]:
e.initScreen()
e.run()

In [1]:
from pgl.pglPipeline import pglRun, pglChoose
r = pglRun.load()
#fullDataPath = pglChoose.getExperimentPath()
#print(fullDataPath)
r.print()

(pglExperiment:load) Loading experimentdata from: /Users/justin/data/randomdots/s0000/session_2026-08-10/run_16-17-35
(pglTask:load) Loading task data from: /Users/justin/data/randomdots/s0000/session_2026-08-10/run_16-17-35/randomDotMotionTask
(pglParameter:load) Loaded parameter dir_coherence from: /Users/justin/data/randomdots/s0000/session_2026-08-10/run_16-17-35/randomDotMotionTask/parameters/dir_coherence
Experiment: randomDots | Subject ID: s0000
Duration: 14s 530ms
Number of volume triggers: 18
Median time between triggers: 0.322s
Mean ± std time between triggers: 0.836 ± 0.688556s
taskName: randomDotMotionTask
Task: Random Dot Motion Task | Trials: 11
Duration=15s 750ms | startTime=2148028.886426917 | endTime=2148044.6371854167
seglen=[1.0, 0.5]
waitUntilVolumeTrigger=[False, False]
coherence=[0.1, 1]
dir=[  0  45  90 135 180 225 270 315]
height=10
width=15
----------------------------------------
dir_coherence
Trial 1 at 0.00s (vol=1): coherence=1, dir=225
Trial 2 at 1.51s (v

In [ ]:
from pgl.pglPipeline import pglChoose
pglChoose()

In [ ]:
pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

pgl.flush()